# Reintegration Readiness Analysis
## Safira Nonprofit — One-Time Feature Importance Study

**Goal:** Identify which resident metrics most strongly predict successful reintegration,
so the social worker modal surfaces the right indicators.  
**Organization:** Safira (fictional nonprofit inspired by Lighthouse Sanctuary)

This is an **explanatory analysis**, not a deployed pipeline. No inference script is produced.

1. **Problem Framing**
2. **Data Acquisition & Feature Engineering**
3. **Exploratory Data Analysis**
4. **Explanatory Modeling** (statsmodels logistic regression — coefficients & significance)
5. **Predictive Feature Importance** (Random Forest permutation importance)
6. **Conclusions — What to Surface in the Modal**

> **Prerequisites:** `pip install -r requirements.txt`  
> **Database:** set `DATABASE_URL` in `../backend/Intex2/Intex2/.env`


## Section 1: Problem Framing

### Business Problem
Social workers manage a caseload of residents across multiple safehouses. Each resident has a
reintegration goal (family reunification, foster care, independent living) but the path to
completion is complex and multi-dimensional.

Currently the modal surfaces a lot of data — but not necessarily the *right* data.
We want to answer:

> *Which tracked metrics most strongly predict whether a resident reaches `reintegration_status = 'Completed'`?*

### Target Variable
**Binary classification:** `reintegrated = 1` if `reintegration_status = 'Completed'`, else `0`.

### Approach: Explanatory First, Predictive Second

| Dimension | This Analysis |
|-----------|---------------|
| **Explanatory** | Logistic Regression (statsmodels) — which features are statistically significant and in which direction |
| **Predictive** | Random Forest — permutation importance ranking of all features |

The output is a **ranked feature list** to guide what we keep on the modal.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import psycopg2
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Modeling
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, classification_report
import statsmodels.api as sm

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('Libraries loaded.')

## Section 2: Data Acquisition & Feature Engineering

In [ ]:
load_dotenv('../backend/Intex2/Intex2/.env')
DATABASE_URL = os.getenv('DATABASE_URL')
conn = psycopg2.connect(DATABASE_URL)
print('Connected to database.')

In [ ]:
# Load all relevant tables
residents       = pd.read_sql('SELECT * FROM residents', conn)
health          = pd.read_sql('SELECT * FROM health_wellbeing_records', conn)
education       = pd.read_sql('SELECT * FROM education_records', conn)
sessions        = pd.read_sql('SELECT * FROM process_recordings', conn)
visitations     = pd.read_sql('SELECT * FROM home_visitations', conn)
plans           = pd.read_sql('SELECT * FROM intervention_plans', conn)

print(f'Residents:    {len(residents)}')
print(f'Health:       {len(health)}')
print(f'Education:    {len(education)}')
print(f'Sessions:     {len(sessions)}')
print(f'Visitations:  {len(visitations)}')
print(f'Plans:        {len(plans)}')

In [ ]:
# ── Target variable ───────────────────────────────────────────────────────────
# Only include residents with a reintegration type (excludes 'None')
df = residents[residents['reintegration_type'].notna() & (residents['reintegration_type'] != 'None')].copy()
df['reintegrated'] = (df['reintegration_status'] == 'Completed').astype(int)

print(f'Residents with reintegration goal: {len(df)}')
print(f'Completed: {df["reintegrated"].sum()} ({df["reintegrated"].mean():.1%})')
print(f'Not completed: {(~df["reintegrated"].astype(bool)).sum()}')

In [ ]:
# ── Risk level encoding ───────────────────────────────────────────────────────
RISK_RANK = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
df['initial_risk_num'] = df['initial_risk_level'].map(RISK_RANK)
df['current_risk_num']  = df['current_risk_level'].map(RISK_RANK)
df['risk_improvement']  = df['initial_risk_num'] - df['current_risk_num']  # positive = improved

# ── Length of stay (parse "X Years Y months" → total months) ─────────────────
def parse_stay_months(s):
    if pd.isna(s): return np.nan
    import re
    y = re.search(r'(\d+)\s*Year', s)
    m = re.search(r'(\d+)\s*month', s)
    return (int(y.group(1)) * 12 if y else 0) + (int(m.group(1)) if m else 0)

df['stay_months'] = df['length_of_stay'].apply(parse_stay_months)

print('Risk and stay features engineered.')

In [ ]:
# ── Health features ───────────────────────────────────────────────────────────
health_scores = health.groupby('resident_id').agg(
    health_avg_score        = ('general_health_score', 'mean'),
    health_latest_score     = ('general_health_score', 'last'),
    nutrition_avg           = ('nutrition_score', 'mean'),
    sleep_avg               = ('sleep_quality_score', 'mean'),
    energy_avg              = ('energy_level_score', 'mean'),
    pct_medical_done        = ('medical_checkup_done', 'mean'),
    pct_dental_done         = ('dental_checkup_done', 'mean'),
    pct_psych_done          = ('psychological_checkup_done', 'mean'),
    health_record_count     = ('general_health_score', 'count'),
).reset_index()

# Health trend: slope of general_health_score over time
def score_trend(grp):
    grp = grp.sort_values('record_date').reset_index(drop=True)
    if len(grp) < 2: return 0.0
    return np.polyfit(range(len(grp)), grp['general_health_score'].fillna(grp['general_health_score'].mean()), 1)[0]

health_trend = health.groupby('resident_id').apply(score_trend).reset_index()
health_trend.columns = ['resident_id', 'health_trend']
health_scores = health_scores.merge(health_trend, on='resident_id', how='left')

print(f'Health features for {len(health_scores)} residents.')

In [ ]:
# ── Education features ────────────────────────────────────────────────────────
edu_features = education.groupby('resident_id').agg(
    avg_attendance_rate     = ('attendance_rate', 'mean'),
    latest_attendance_rate  = ('attendance_rate', 'last'),
    avg_progress_pct        = ('progress_percent', 'mean'),
    latest_progress_pct     = ('progress_percent', 'last'),
    education_record_count  = ('attendance_rate', 'count'),
).reset_index()

# Attendance trend
def attendance_trend(grp):
    grp = grp.sort_values('record_date').reset_index(drop=True)
    if len(grp) < 2: return 0.0
    return np.polyfit(range(len(grp)), grp['attendance_rate'].fillna(grp['attendance_rate'].mean()), 1)[0]

edu_trend = education.groupby('resident_id').apply(attendance_trend).reset_index()
edu_trend.columns = ['resident_id', 'attendance_trend']
edu_features = edu_features.merge(edu_trend, on='resident_id', how='left')

print(f'Education features for {len(edu_features)} residents.')

In [ ]:
# ── Counseling session features ───────────────────────────────────────────────
session_features = sessions.groupby('resident_id').agg(
    session_count           = ('session_date', 'count'),
    pct_progress_noted      = ('progress_noted', 'mean'),
    pct_concerns_flagged    = ('concerns_flagged', 'mean'),
    avg_session_duration    = ('session_duration_minutes', 'mean'),
).reset_index()

print(f'Session features for {len(session_features)} residents.')

In [ ]:
# ── Home visitation features ──────────────────────────────────────────────────
visitations['is_cooperative'] = (visitations['family_cooperation_level'] == 'Cooperative').astype(float)
visitations['has_safety_concern'] = visitations['safety_concerns_noted'].astype(float)

visit_features = visitations.groupby('resident_id').agg(
    visit_count             = ('visit_date', 'count'),
    pct_cooperative         = ('is_cooperative', 'mean'),
    pct_safety_concerns     = ('has_safety_concern', 'mean'),
).reset_index()

print(f'Visitation features for {len(visit_features)} residents.')

In [ ]:
# ── Intervention plan features ────────────────────────────────────────────────
plans['is_achieved'] = (plans['status'] == 'Achieved').astype(float)

plan_features = plans.groupby('resident_id').agg(
    plan_count              = ('plan_id', 'count'),
    pct_plans_achieved      = ('is_achieved', 'mean'),
).reset_index()

print(f'Plan features for {len(plan_features)} residents.')

In [ ]:
# ── Merge all features ────────────────────────────────────────────────────────
feat = df[[
    'resident_id', 'reintegrated',
    'initial_risk_num', 'current_risk_num', 'risk_improvement',
    'stay_months',
    'case_category', 'reintegration_type', 'referral_source',
    'is_pwd', 'has_special_needs',
    'family_is_4ps', 'family_solo_parent', 'family_indigenous',
    'family_parent_pwd', 'family_informal_settler',
]].copy()

for df_features, key in [
    (health_scores,    'resident_id'),
    (edu_features,     'resident_id'),
    (session_features, 'resident_id'),
    (visit_features,   'resident_id'),
    (plan_features,    'resident_id'),
]:
    feat = feat.merge(df_features, on=key, how='left')

print(f'Final dataset: {feat.shape[0]} residents × {feat.shape[1]} columns')
print(f'Missing values per column:\n{feat.isnull().sum()[feat.isnull().sum() > 0]}')

## Section 3: Exploratory Data Analysis

In [ ]:
# Class balance
fig, ax = plt.subplots(figsize=(5, 3))
feat['reintegrated'].value_counts().plot(kind='bar', ax=ax, color=['#ef4444', '#22c55e'], edgecolor='white')
ax.set_xticklabels(['Not Completed', 'Completed'], rotation=0)
ax.set_title('Reintegration Status Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('reintegration_class_balance.png', dpi=150)
plt.show()

In [ ]:
# Key numeric features by reintegration outcome
numeric_features = [
    'risk_improvement', 'stay_months',
    'health_avg_score', 'health_trend',
    'avg_attendance_rate', 'attendance_trend',
    'pct_progress_noted', 'pct_concerns_flagged',
    'pct_cooperative', 'pct_safety_concerns',
    'pct_plans_achieved',
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    if col not in feat.columns: continue
    for label, color in [(0, '#ef4444'), (1, '#22c55e')]:
        subset = feat[feat['reintegrated'] == label][col].dropna()
        axes[i].hist(subset, alpha=0.6, color=color, bins=15,
                     label='Completed' if label else 'Not Completed', density=True)
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)

for j in range(i + 1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Feature Distributions by Reintegration Outcome', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('reintegration_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Point-biserial correlations with target
correlations = {}
for col in numeric_features:
    if col not in feat.columns: continue
    valid = feat[[col, 'reintegrated']].dropna()
    if len(valid) < 10: continue
    r, p = stats.pointbiserialr(valid['reintegrated'], valid[col])
    correlations[col] = {'correlation': r, 'p_value': p}

corr_df = pd.DataFrame(correlations).T.sort_values('correlation', key=abs, ascending=False)
print('Correlation with reintegration_completed:')
print(corr_df.round(4))

In [ ]:
# Correlation bar chart
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#22c55e' if v >= 0 else '#ef4444' for v in corr_df['correlation']]
bars = ax.barh(corr_df.index, corr_df['correlation'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Point-Biserial Correlation with Reintegration')
ax.set_title('Feature Correlation with Reintegration Completion')

# Mark significance
for i, (idx, row) in enumerate(corr_df.iterrows()):
    if row['p_value'] < 0.05:
        ax.text(row['correlation'] + (0.01 if row['correlation'] >= 0 else -0.01),
                i, '*', va='center', fontsize=12,
                ha='left' if row['correlation'] >= 0 else 'right')

ax.text(0.99, 0.01, '* p < 0.05', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('reintegration_correlations.png', dpi=150)
plt.show()

## Section 4: Explanatory Modeling (Statsmodels Logistic Regression)

Logistic regression with statsmodels gives us **coefficients, p-values, and odds ratios** —
telling us not just *what* matters but *how much* and in *which direction*.

In [ ]:
# Use numeric features only for the explanatory model
explanatory_features = [
    'risk_improvement', 'current_risk_num', 'stay_months',
    'health_avg_score', 'health_trend',
    'avg_attendance_rate', 'attendance_trend',
    'pct_progress_noted', 'pct_concerns_flagged',
    'pct_cooperative', 'pct_safety_concerns',
    'pct_plans_achieved', 'pct_psych_done',
]
explanatory_features = [f for f in explanatory_features if f in feat.columns]

expl_data = feat[explanatory_features + ['reintegrated']].dropna()
print(f'Explanatory model sample: {len(expl_data)} residents')

X_expl = expl_data[explanatory_features]
y_expl = expl_data['reintegrated']

# Standardize for comparable coefficients
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_expl), columns=explanatory_features)
X_with_const = sm.add_constant(X_scaled)

logit_model = sm.Logit(y_expl.values, X_with_const.values)
result = logit_model.fit(maxiter=200, disp=False)
print(result.summary2())

In [ ]:
# Odds ratios with confidence intervals
params = result.params[1:]  # drop const
conf   = result.conf_int()[1:]
pvals  = result.pvalues[1:]

odds_df = pd.DataFrame({
    'feature':    explanatory_features,
    'odds_ratio': np.exp(params),
    'ci_low':     np.exp(conf[:, 0]),
    'ci_high':    np.exp(conf[:, 1]),
    'p_value':    pvals,
    'significant': pvals < 0.05,
}).sort_values('odds_ratio', ascending=True)

print('\nOdds Ratios (standardized features):')
print(odds_df[['feature', 'odds_ratio', 'p_value', 'significant']].round(3).to_string(index=False))

In [ ]:
# Odds ratio forest plot
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#22c55e' if s else '#94a3b8' for s in odds_df['significant']]

ax.barh(odds_df['feature'], odds_df['odds_ratio'] - 1,
        left=1, color=colors, edgecolor='white', alpha=0.85)
ax.errorbar(
    odds_df['odds_ratio'], range(len(odds_df)),
    xerr=[odds_df['odds_ratio'] - odds_df['ci_low'],
          odds_df['ci_high'] - odds_df['odds_ratio']],
    fmt='none', color='black', capsize=3, linewidth=1
)
ax.axvline(1, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Odds Ratio (1 SD change in feature)')
ax.set_title('Reintegration Completion — Explanatory Odds Ratios\n(green = statistically significant p<0.05)')
plt.tight_layout()
plt.savefig('reintegration_odds_ratios.png', dpi=150)
plt.show()

## Section 5: Predictive Feature Importance (Random Forest)

Random Forest captures non-linear relationships and interactions that logistic regression misses.
Permutation importance tells us how much each feature contributes to predictive accuracy.

In [ ]:
# Full feature set (numeric + encoded categoricals)
all_numeric = [
    'initial_risk_num', 'current_risk_num', 'risk_improvement', 'stay_months',
    'health_avg_score', 'health_latest_score', 'health_trend',
    'nutrition_avg', 'sleep_avg', 'energy_avg',
    'pct_medical_done', 'pct_dental_done', 'pct_psych_done',
    'health_record_count',
    'avg_attendance_rate', 'latest_attendance_rate', 'attendance_trend',
    'avg_progress_pct', 'latest_progress_pct', 'education_record_count',
    'session_count', 'pct_progress_noted', 'pct_concerns_flagged', 'avg_session_duration',
    'visit_count', 'pct_cooperative', 'pct_safety_concerns',
    'plan_count', 'pct_plans_achieved',
    'is_pwd', 'has_special_needs',
    'family_is_4ps', 'family_solo_parent', 'family_indigenous',
    'family_parent_pwd', 'family_informal_settler',
]
all_numeric = [f for f in all_numeric if f in feat.columns]

categorical = ['case_category', 'reintegration_type', 'referral_source']
categorical = [f for f in categorical if f in feat.columns]

model_data = feat[all_numeric + categorical + ['reintegrated']].dropna(subset=['reintegrated'])
print(f'Predictive model sample: {len(model_data)} residents')

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, all_numeric),
    ('cat', categorical_transformer, categorical),
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )),
])

X = model_data[all_numeric + categorical]
y = model_data['reintegrated']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(rf_pipeline, X, y, cv=cv,
                            scoring=['roc_auc', 'f1_weighted'],
                            return_train_score=True)

print(f'CV AUC-ROC:  {cv_results["test_roc_auc"].mean():.3f} ± {cv_results["test_roc_auc"].std():.3f}')
print(f'CV F1:       {cv_results["test_f1_weighted"].mean():.3f} ± {cv_results["test_f1_weighted"].std():.3f}')

In [ ]:
# Fit on full data for permutation importance
rf_pipeline.fit(X, y)
X_transformed = rf_pipeline['preprocessor'].transform(X)

# Get feature names after encoding
cat_feature_names = rf_pipeline['preprocessor'].named_transformers_['cat']['encoder'].get_feature_names_out(categorical).tolist()
all_feature_names = all_numeric + cat_feature_names

perm_imp = permutation_importance(
    rf_pipeline['model'], X_transformed, y,
    n_repeats=30, random_state=42, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature':    all_feature_names,
    'importance': perm_imp.importances_mean,
    'std':        perm_imp.importances_std,
}).sort_values('importance', ascending=False)

print('Top 20 features by permutation importance:')
print(perm_df.head(20).round(4).to_string(index=False))

In [ ]:
# Permutation importance chart — top 20
top20 = perm_df.head(20).sort_values('importance')

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top20['feature'], top20['importance'],
        xerr=top20['std'], color='#3b82f6', edgecolor='white', alpha=0.85, capsize=3)
ax.set_xlabel('Mean Decrease in AUC-ROC (permutation importance)')
ax.set_title('Top 20 Features — Reintegration Readiness\n(Random Forest Permutation Importance)')
plt.tight_layout()
plt.savefig('reintegration_feature_importance.png', dpi=150)
plt.show()

## Section 6: Conclusions — What to Surface in the Modal

In [ ]:
# Combined ranking: average rank across correlation, odds ratio effect size, and permutation importance
core_features = [
    'risk_improvement', 'current_risk_num',
    'pct_cooperative', 'pct_safety_concerns',
    'pct_progress_noted', 'pct_concerns_flagged',
    'avg_attendance_rate', 'attendance_trend',
    'health_avg_score', 'health_trend',
    'pct_plans_achieved',
    'pct_psych_done', 'stay_months',
]

# Rank from permutation importance
perm_rank = perm_df[perm_df['feature'].isin(core_features)][['feature', 'importance']].copy()
perm_rank['perm_rank'] = perm_rank['importance'].rank(ascending=False)

# Rank from correlation
corr_rank = corr_df[corr_df.index.isin(core_features)][['correlation']].copy()
corr_rank['corr_rank'] = corr_rank['correlation'].abs().rank(ascending=False)
corr_rank.index.name = 'feature'
corr_rank = corr_rank.reset_index()

summary = perm_rank.merge(corr_rank, on='feature', how='outer')
summary['avg_rank'] = summary[['perm_rank', 'corr_rank']].mean(axis=1)
summary = summary.sort_values('avg_rank')

print('\n=== RECOMMENDED MODAL INDICATORS (ranked by combined evidence) ===')
print(summary[['feature', 'importance', 'correlation', 'avg_rank']].round(3).to_string(index=False))

In [ ]:
# Final summary chart
summary_top = summary.head(10).sort_values('avg_rank', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(summary_top['feature'], summary_top['importance'],
               color='#3b82f6', edgecolor='white', alpha=0.85)
ax.set_xlabel('Permutation Importance')
ax.set_title('Top Indicators to Surface in the Resident Modal\n(combined evidence from correlation + random forest)')
plt.tight_layout()
plt.savefig('reintegration_modal_recommendations.png', dpi=150)
plt.show()

print('\nDone. Charts saved to ml-pipelines/.')

In [ ]:
conn.close()
print('Database connection closed.')